# CustomWear E-Commerce Sales

## Table of Contents <a id='back'></a>
- [Project Introduction](#project-introduction)
    - [Analysis Objectives](#analysis-objectives)
- [Importing Libraries and Opening Data Files](#importing-libraries-and-opening-data-files)
- [Data Wrangling](#data-wrangling)
    - [Duplicates](#duplicates)
    - [Missing Values](#missing-values)
    - [Data Transformation](#data-transformation)

## Project Introduction

This project represents the first phase of an end-to-end data analytics workflow and focuses on preparing a real-world business dataset for analysis using Python. The dataset was provided by a former client who requested that both their identity and any identifying business information remain anonymous. To preserve confidentiality, all references throughout this project have been generalized while maintaining the integrity and analytical value of the dataset.

The client operates in the fashion apparel customization industry, specializing in the application of custom decals and embroidery to a variety of apparel products. The dataset contains operational information related to customer orders and production activities.

Prior to beginning any analysis, all personally identifiable information (PII) and other sensitive customer data were proactively removed to protect customer privacy. This includes customer names, street addresses, phone numbers, email addresses, payment and credit card information, and any other fields that could directly or indirectly identify individual customers. These modifications ensure that the dataset follows data privacy best practices while remaining suitable for analytical purposes.

Like many operational datasets, the raw data contains inconsistencies, missing values, duplicate records, and formatting issues that must be addressed before meaningful analysis can be performed. This project documents the data cleaning and preprocessing steps taken to transform the raw dataset into a reliable, analysis-ready resource for the subsequent stages of the analytics process.

### Analysis Objectives

The primary objective of this phase is to clean and preprocess the dataset using Python. The data cleaning process is intended to improve data quality by identifying and resolving issues that could negatively impact downstream analysis.

The specific objectives of this phase are to:

 - Assess the overall quality and structure of the dataset
  - Remove duplicate records where appropriate
 - Identify and address missing, incomplete, or inconsistent data
 - Correct data types and standardize data formats
 - Validate the cleaned dataset to ensure accuracy and consistency
 - Produce a reliable, analysis-ready dataset for exploratory data analysis (EDA), visualization, and future business intelligence reporting

Successfully completing these objectives establishes a dependable foundation for the remaining phases of the project, allowing subsequent analyses to generate accurate insights and support data-driven decision-making.


[Back to Table of Contents](#back)

## Importing Libraries and Opening Data Files

In [ ]:
# Importing the needed libraries for this assignment
import pandas as pd

In [ ]:
# Importing file for assignment
try:
    df = pd.read_csv('xxxecom_orders_data.csv', sep=',')
except:
    df = pd.read_csv('/datasets/xxxecom_orders_data.csv', sep=',')

try:
    df = pd.read_csv('xxxecom_orders_data.csv', sep=',')
except:
    df = pd.read_csv('/datasets/xxxecom_orders_data.csv', sep=',')

[Back to Table of Contents](#back)

## Data Wrangling

In [ ]:
# Sampling the data
df.info()
df.head()

Observation:

- The order_id and customer_id column are float data types and could be converted int data type to lower data usage

- Across all the string data type columns are in mixed lowercase and proper case format and should all be converted to snake case format for consistency and remove possible leading or trailing white spaces

- The order_date and ship_date columns are in string and should be converted to datetime format to lower data usage

### Duplicates

In [ ]:
# Checking for duplicates
df.duplicated().sum()

Observation:

- There are no repeat rows which is good 

- Since the order_id column is the unique identifier I will check that it also has no duplicates

In [ ]:
# Counting number of possible duplicated rows
print(f'{df['order_id'].duplicated().sum()} duplicate order_id')

# Counting the unique identifier column for unqiue values
print(f'{df['order_id'].nunique()} unique values')

Observation:

- However, when looking into the unique identifier column there are only 619 unique values and 367 duplicate order_id values

- One possibility is that some orders could repeat if there are multiple different products in the same order but this could still be a problem and needs further inspection

In [ ]:
# Looking more into the order_id column
df[df['order_id'].duplicated(keep=False)].sort_values(by='order_id').head(30)

Observation:

- Looking at the first 30 rows many order_id values repeat as many as 3 times

- Despite having the same order_id the age, address locations, and order dates are different indicating that these should be different orders

- Additionally, there is a null value in the customer_id column indicating that later I need to fix null values

In [ ]:
# Replacing all order_id values to make them unique
df['order_id'] = range(1, len(df) + 1)

# Counting number of possible duplicated rows
print(f'{df['order_id'].duplicated().sum()} duplicate order_id')
print(df['order_id'].head())

Observation:

- Removed all order_id duplicate values

In [ ]:
# Looking into duplicate customer_id values
df['customer_id'].duplicated().sum()

Observation:

- There are 370 duplicate customer_id values which may need to be replaced or fixed

- These could be repeat customers so I need to be careful of which values to remove

In [ ]:
# Looking at customer_id column for unique customers
df[df['customer_id'].duplicated(keep=False)].sort_values(by='customer_id').head(30)

Observation:

- Looking more into the customer_id duplicates it shows each row has the same age, city, and state which means its a different order from the same customer

- No fixes are needed for this column and the duplicates can remain the same

[Back to Table of Contents](#back)

### Missing Values

In [ ]:
# Checking for null values
df.isna().sum()

Observation:

- Looking into the null values I need to look into the customer_id, order_date, and shipping date columns

In [ ]:
# Sampling the customer_id column
df[df['customer_id'].isna()]

Observation:

- Since there are only 22 null values I can remove since its only 2.2% of the total rows but I want to save as many of them as I can

- I will assign new customer id values to these null columns to retain as much sales data as I can since these are all real orders

In [ ]:
# Identifying the max customer id value to add new values after it
start = int(df['customer_id'].max()) + 1

# Replacing null customer id values with new values 
df.loc[df['customer_id'].isna(), 'customer_id'] = range(start, start + df['customer_id'].isna().sum())

# Sampling the new data to see if changes were executed
print(f'{df['customer_id'].isna().sum()} null values')
print(f'{df['customer_id'].max()} max customer_id')

Observation:

- Retained all sales orders as new customer_id values

In [ ]:
# Sampling the order_date column
df[df['order_date'].isna()]

Observation:

- It appears that all the order date and shipping date values are missing together

- For this missing data it would be better to remove it since these could be considered missing or canceled orders

In [ ]:
# Since the order and shipping dates are correlated I just need to remove all the order_date null values and the shipping dates will be removed with them
df = df.dropna(subset=['order_date'])

# Checking for null values
df.isna().sum()

Observation:

- All null values are removed

[Back to Table of Contents](#back)

### Data Transformation

In [ ]:
# Getting general information about the dataset
df.info()
df.head()

Observation:

- Based on the meta data there are several improvements that can be made:

    1) The column names and the table elements can be converted to snakecase format for consistency and readability

    2) The customer_id column can be converted to an integer data type to reduce data usage

    3) The date columns can be converted to datetime format

In [ ]:
# Checking for snakecase format
df.columns

In [ ]:
# Renaming column names to snake_case format
df = df.rename(columns={'Quantity': 'quantity'})

df.columns

In [ ]:
# Converting string data value to datetime data type
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# Converting the customer_id column from float to int to reduce data usage
df['customer_id'] = df['customer_id'].astype('int')

df.info()

In [ ]:
# Converting all elements into snakecase format and removing all nonlegible characters
for column in df.columns:
    if df[column].dtype == 'object':
        df[column] = df[column].str.lower()
        df[column] = df[column].str.replace(' ', '_')

df.head()

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['city'].unique()

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['state'].unique()

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['product_category'].unique()

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['sub_category'].unique()

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['color'].unique()

Observation:

- All string columns values seem to have no apparent spelling errors

In [ ]:
# Deriving more columns from current data to have more options later for analysis
df['total_sale'] = df['price'] * df['quantity']
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day_name()
df['order_quarter'] = df['order_date'].dt.quarter
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 45, 55, 65],
    labels=[
        '18-25',
        '26-35',
        '36-45',
        '46-55',
        '56-65'])

df.info()
df.head()

In [ ]:
# Saving the new cleaned dataset
df.to_csv('cleaned_customewear_data.csv', index=False) 

[Back to Table of Contents](#back)